<a href="https://colab.research.google.com/github/konderal333/HGT-2022-EmDomArDon/blob/main/bert2bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load models and data

In [ ]:
!git clone https://github.com/konderal333/HGT-2022-EmDomArDon.git

In [ ]:
import numpy as np
import pandas as pd
import re

In [ ]:
df = pd.read_csv('/content/HGT-2022-EmDomArDon/cikkek_10k_cleanedv1.csv')

In [ ]:
print(df.isnull().values.any())
print(sum(df.duplicated()))

In [ ]:
df = df.dropna()
df = df.drop_duplicates()

In [ ]:
df = df.apply(lambda x: x.str.strip())
df['Textbody'] = df['Textbody'].apply(lambda x: re.sub(r"(\.)([A-Z])", r'\1 \2', str(x))) # add missing whitespace between sentences

df['Lead'] = df['Title'] + '. ' + df['Headline'] # maybe only headline and drop title?
df = df.drop(['Title', 'Headline'],axis=1)
df.head()

#reindexing the dataframe
df.reset_index(inplace=True)
df = df.drop(['index'], axis=1)

In [ ]:
%%capture
!pip install transformers

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('SZTAKI-HLT/hubert-base-cc')

In [ ]:
# bert2bert
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1jVivzwtgoZx38yMeZA5BthGGvoPxBZk5' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1jVivzwtgoZx38yMeZA5BthGGvoPxBZk5" -O model_bert2bert.tar.gz && rm -rf /tmp/cookies.txt

# model
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1GEdOmmKZEJOD2Ei28FtVEf534-nBEcBd' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1GEdOmmKZEJOD2Ei28FtVEf534-nBEcBd" -O model.tar.gz && rm -rf /tmp/cookies.txt

In [ ]:
import shutil
shutil.unpack_archive('model.tar.gz', 'model')
shutil.unpack_archive('model_bert2bert.tar.gz', 'bert2bert')

In [ ]:
import torch
from transformers import EncoderDecoderModel

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

model_path = '/content/model/model/pytorch_model.bin'
config = '/content/model/model/config.json'
model = EncoderDecoderModel.from_pretrained(model_path, config=config).to(device)


# bert2bert_model_path = '/content/bert2bert/model/pytorch_model.bin'
# bert2bert_config = '/content/bert2bert/model/config.json'
# bert2bert = EncoderDecoderModel.from_pretrained(bert2bert_model_path, config=bert2bert_config).to(device)


Tutorial notebook:

https://colab.research.google.com/drive/1WIk2bxglElfZewOHboPFNj8H44_VAyKE?usp=sharing&fbclid=IwAR2-pdnudSutvgqIlUUg4NP1Q0wJO9NQbteg_wVL9n8gsS2yMTfMp9YfC6w#scrollTo=bYmdx-W1NAky

The majority of the codes found in that notebook is outdated, so we needed to make a couple of augmentations and changes in order to make them work in our environment.

##Functions to handle inputs larger than 512 tokens

In [ ]:
import re
import nltk
nltk.download('punkt')

<p>Method to reduce the length of sentences with more than 510 tokens:
<p>We get the first few whole sentences until we reach 255 tokens. (Let us assume that we get n tokens from these sentences, where n&#60;255). Then we get the last few whole sentences until these reach the remaining limit of tokens (510-n).
<p>So with this method we reduce the longer articles to texts which contain 510 tokens (or a bit less than 510), so that we get the beginning and the end of the article.

In [ ]:
def text_to_token_list(art_title, x):      
  token_list = []
  sen_list = []

  x = re.sub(r"(\.)([A-Z])", r'\1 \2', x) # fixes missings whitespaces between sentences
 
  for sen in nltk.tokenize.sent_tokenize(x):
    token_list.append(tokenizer.tokenize(sen))
    sen_list.append(sen)

  return token_list, sen_list

In [ ]:
dict_of_tokens = {} 
#this dict will contain the article title as keys and the tokenized lists from the function below as values for other purposes

def text_to_tokens(art_title, x):      
  #insert the text to tokenize; first breaks text into sentences, then makes a list of lists with the i-th element being the tokenization of the i-th sentence
  token_list = []

  x = re.sub(r"(\.)([A-Z])", r'\1 \2', x) # fixes missings whitespaces between sentences

  tokens, sentences = text_to_token_list(art_title, x)  #tokens and sentences

  new_text = ''      #this is the text which contains less than (or equal to) 510 tokens

  #total tokennumber
  len1 = 0
  for i in tokens:
    for j in i:
      len1 += 1

  #if the text is > 510 tokens, we will get the first few whole sentences (~255 tokens), and the last few whole sentences (~255 tokens)
  if len1 > 510:
    begin_sum = 0
    begin_tokens = []
    for index, elem1 in enumerate(tokens):
      if begin_sum + len(elem1) > 255:
        break
      else:
        begin_sum += len(elem1)
        begin_tokens.append(elem1)
        new_text = new_text + sentences[index] + " "   #appending the new text  

    end_indexes = []
    end_sum = 0
    end_tokens = []
    tokens.reverse()
    for index, elem2 in enumerate(tokens):
      if end_sum + len(elem2) > 510-begin_sum:
        break
      else:
        end_sum += len(elem2)
        end_tokens.append(elem2)
        end_indexes.append(-index-1)

    #finally reconstructing the text which has less than (or equal to) 510 tokens 
    end_indexes.reverse()
    for end_index in end_indexes:
      new_text = new_text + sentences[end_index] + " "


    end_tokens.reverse()
    final_tokens = []
    for elem3 in begin_tokens:
      final_tokens.append(elem3)
    for elem4 in end_tokens:
      final_tokens.append(elem4)

    dict_of_tokens[art_title] = final_tokens
    return new_text.strip()

  #if the text is < 510 tokens, we do not have to get the beginning and the end, we can use the whole text
  else:
    for sen in nltk.tokenize.sent_tokenize(x):
      new_text = new_text + sen + " "
      token_list.append(tokenizer.tokenize(sen))

    dict_of_tokens[art_title] = token_list
    return new_text.strip()

In [ ]:
# making the BERT tokenized sentences and the reduced sentences from each article

for i in range(len(df.Lead)):
  new_text = text_to_tokens(df.Lead[i].strip(), df.Textbody[i])
  df.at[i, 'Textbody'] = new_text

In [ ]:
import matplotlib.pyplot as plt
import statistics

number_of_tokens_all = []

for tokens in dict_of_tokens.values():
  length = 0
  for lists in tokens:
    length += len(lists)
  number_of_tokens_all.append(length)

In [ ]:
len(dict_of_tokens.keys())

In [ ]:
plt.figure(figsize=(8,6), dpi=70)
plt.hist(number_of_tokens_all, bins=100, density=False)
#plt.title('Distribution of number of tokens generated from articles')
plt.xlabel('Number of tokens', fontsize=12)
plt.ylabel('Number of articles', fontsize=12)
plt.show()

In [ ]:
# number of articles with tokens more that 510:
print(np.sum(np.asarray(number_of_tokens_all) > 510))

In [ ]:
# checking if the new texts really has the appropriate number of tokens
for index, row in df.iterrows():
  x = nltk.tokenize.sent_tokenize(row['Textbody'])
  all_len = 0
  for elem in x:
    all_len += len(tokenizer.tokenize(elem))
  if all_len > 510:
    print(index)

## Process data

In [ ]:
def process_data_to_model_inputs(batch):
        # Tokenize the input and target data
        inputs = tokenizer(batch['Textbody'], padding='max_length', truncation=True, max_length=512)
        outputs = tokenizer(batch['Lead'], padding='max_length', truncation=True, max_length=512)

        batch['input_ids'] = inputs.input_ids
        batch['attention_mask'] = inputs.attention_mask
        batch['decoder_input_ids'] = outputs.input_ids
        batch['decoder_attention_mask'] = outputs.attention_mask
        batch['labels'] = outputs.input_ids.copy()

        #batch['labels'] = [[-100 if token == tokenizer.pad_token_id else token for token in labels]
        #                    for labels in batch['labels']]
        batch['labels'] = [-100 if token == tokenizer.pad_token_id else token for token in batch['labels']]

        return batch

In [ ]:
#number of articles
df.shape[0]

In [ ]:
# Full Data

# split is 70-15-15

train_data = df.iloc[0:int(np.floor(df.shape[0]*0.7))]
val_data = df.iloc[int(np.floor(df.shape[0]*0.7)):int(np.floor(df.shape[0]*0.85))]
test_data = df.iloc[int(np.floor(df.shape[0]*0.85)):]


#Partial Data
#train_data = df.iloc[0:20]
#val_data = df.iloc[20:30]
#test_data = df.iloc[30:40]

#train_data.reset_index(inplace=True)
#val_data.reset_index(inplace=True)
#test_data.reset_index(inplace=True)

#train_data = train_data.drop(['index'], axis=1)
#val_data = val_data.drop(['index'], axis=1)
#test_data = test_data.drop(['index'], axis=1)

In [ ]:
#Transforming the train and validation data
train_data = train_data.apply(process_data_to_model_inputs, axis=1)
train_data = train_data.drop(['Textbody', 'Lead'], axis=1)

val_data = val_data.apply(process_data_to_model_inputs, axis=1)
val_data = val_data.drop(['Textbody', 'Lead'], axis=1)

In [ ]:
train_data.head()

In [ ]:
%%capture
!pip install datasets

In [ ]:
# converting to arrow dataset (easier to use)
import pyarrow as pa
import pyarrow.dataset as ds
from datasets import Dataset

train_data = Dataset(pa.Table.from_pandas(train_data))
val_data = Dataset(pa.Table.from_pandas(val_data))
test_data = Dataset(pa.Table.from_pandas(test_data))

# format type
train_data

In [ ]:
#converting the train and validation sets to tensors
train_data.set_format(
    type="torch", columns=["input_ids", "attention_mask", "decoder_input_ids", "decoder_attention_mask", "labels"],
)

val_data.set_format(
    type="torch", columns=["input_ids", "attention_mask", "decoder_input_ids", "decoder_attention_mask", "labels"],
)

In [ ]:
# example
train_data["labels"]

## Prediction and evaluation before training

In [ ]:
# Metrics
!pip install rouge_score
import datasets
rouge = datasets.load_metric("rouge")

In [ ]:
# Summary generation

def generate_summary(batch):
    # cut off at BERT max length 512
    inputs = tokenizer(batch['Textbody'], padding='max_length', truncation=True, max_length=512, return_tensors='pt')
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)

    outputs = model.generate(input_ids, attention_mask=attention_mask, max_length=142)

    output_str = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    batch['pred_summary'] = output_str

    return batch


In [ ]:
# Example article
batch = df.iloc[10]
batch['Textbody']

In [ ]:
# Now the summary
inputs = tokenizer(batch['Textbody'], padding='max_length', truncation=True, max_length=512, return_tensors='pt')
input_ids = inputs.input_ids.to(device)
attention_mask = inputs.attention_mask.to(device)
outputs = model.generate(input_ids, attention_mask=attention_mask, max_length=50)
output_str = tokenizer.batch_decode(outputs, skip_special_tokens=True)
output_str[0]

In [ ]:
# Original lead
df.iloc[10]['Lead']

In [ ]:
# let's see the tokens
for t in input_ids:
    print(tokenizer.convert_ids_to_tokens(t))

In [ ]:
# Now evaluate the whole test set
batch_size = 64

results = test_data.map(generate_summary, batched=True, batch_size=batch_size)
scores = rouge.compute(predictions=results["pred_summary"], references=results["Lead"], rouge_types=["rouge1", "rouge2", "rougeL"])
print(scores['rouge1'].mid)
print(scores['rouge2'].mid)
print(scores['rougeL'].mid)

In [ ]:
# let's see a couple examples
print(results['Lead'][3])
print(results['pred_summary'][3])
print("")
print(results['Lead'][5])
print(results['pred_summary'][5])
print("")
print(results['Lead'][9])
print(results['pred_summary'][9])

As we can see, the sentences are mostly correct grammatically, but their meaning leaves a lot to be desired.

## Train further

In [ ]:
# Metrics
!pip install rouge_score
import datasets
rouge = datasets.load_metric("rouge")

In [ ]:
#next step: ezeket ?ll?tani
model.config.max_length = 128
model.config.min_length = 56
model.config.no_repeat_ngram_size = 3
model.config.early_stopping = True
model.config.length_penalty = 2.0
model.config.num_beams = 3

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

batch_size = 20

training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy="steps",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    fp16=True, 
    output_dir="./output",
    generation_max_length=128,
    logging_steps=16,
    save_steps=48,
    eval_steps=16,
    gradient_accumulation_steps=1, 
    eval_accumulation_steps=None, 
    eval_delay=0, 
    learning_rate=5e-07, 
    weight_decay=0.01,
    load_best_model_at_end=True
)




In [ ]:
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = tokenizer.pad_token_id
    label_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    rouge_output = rouge.compute(predictions=pred_str, references=label_str, rouge_types=["rouge1"])["rouge1"].mid

    return {
        "rouge1_precision": round(rouge_output.precision, 4),
        "rouge1_recall": round(rouge_output.recall, 4),
        "rouge1_fmeasure": round(rouge_output.fmeasure, 4),
    }

In [ ]:
# instantiate trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_data,
    eval_dataset=val_data,
)
trainer.train()

In [ ]:
# load the partially trained model
model = EncoderDecoderModel.from_pretrained("/content/output/checkpoint-960").to(device)

## Evaluation after further training

In [ ]:
batch_size = 64

results = train_data.map(generate_summary, batched=True, batch_size=batch_size)
scores = rouge.compute(predictions=results["pred_summary"], references=results["Lead"], rouge_types=["rouge1", "rouge2", "rougeL"])
print(scores['rouge1'].mid)
print(scores['rouge2'].mid)
print(scores['rougeL'].mid)

In [ ]:
# save to csv
results_df = pd.DataFrame({})
results_df['Lead'] = results['Lead']
results_df['pred_summary'] = results['pred_summary']
results_df['Textbody'] = results['Textbody']
results_df.to_excel('results_df.xlsx', index=False)